# kaggle model server — studio

run all → click the printed `gradio.live` link → everything is tabs:

- **Launch** — pick an llm + quant (with sizes), launch/stop, live log tail
- **Chat** — streaming chat with whatever model is up
- **Image** — load a diffusers model on gpu 1 (coexists with the llm), prompt → png
- **Video** — one button boots headless comfyui and hands you its full node-gui url

the login is printed by the cell below (random password each session). the llm also gets its own cloudflared url for api use — shown in the Launch tab's status box.

In [ ]:
# --- setup: pull in the shared modules -----------------------------------
!rm -rf /kaggle/working/harness_src && git clone https://github.com/Meru143/kaggle-model-server.git /kaggle/working/harness_src
import sys
sys.path.append("/kaggle/working/harness_src")

# gradio pinned to the major this studio is tested against -- kaggle
# preinstalls an older one whose ui behaves differently (bare Error
# pills, tuple chat history, Number rendering blanks as 0)
!pip install -q huggingface_hub requests && pip install -q -U "gradio>=6,<7"

In [ ]:
import importlib, os, secrets, sys

# gated models (flux1-dev, ideogram-4): kaggle secrets are NOT env vars --
# export the attached HF_TOKEN secret so gated downloads authenticate
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    pass  # no secret attached -- fine for everything ungated

# re-running setup + this cell picks up freshly-pulled code without a
# kernel restart (python caches imports; plain re-import returns stale code)
if "harness" in sys.modules:
    try:
        sys.modules["harness"].stop()  # old module owns any running procs
    except Exception:
        pass
for _m in ("model_registry", "harness", "image_models", "comfy_bootstrap", "sdcpp", "control_panel"):
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

from control_panel import launch_panel

user, password = "joy", secrets.token_urlsafe(8)
print(f"studio login -> user: {user}   password: {password}")
launch_panel(auth=(user, password))  # prints the public gradio.live link

leave this notebook running — the studio dies with the session. `stop()` from `harness` (or the studio's Stop button) tears the model down; server log lives at `/kaggle/tmp/llama-server.log`.